# 🚗 Training & Benchmark Pipeline: Deteksi Pedestrian Low-Light (YOLOv8)

> **Mata Kuliah:** Visi Komputer (IF0PP006) — Semester Ganjil 2026/2027  
> **Kelompok:** 9 (Krisantus Linardi & Kristofer Leonard)  
> **Platform:** Google Colab (GPU NVIDIA T4)

---

## 📌 Ringkasan 4 Eksperimen Terkontrol

| Eksperimen | Model | Dataset / Prapemrosesan Training | Evaluasi Split |
|:---:|---|---|---|
| **B0** | `yolov8n.pt` (COCO) | Zero-Shot (Tanpa Training Tambahan) | Citra Validasi Malam Asli |
| **Exp-A** | YOLOv8n Fine-Tuned | Dataset Low-Light Standar | Citra Validasi Malam Asli |
| **Exp-B** | YOLOv8n Fine-Tuned | Dataset Low-Light + **L-channel CLAHE** | Citra Validasi Malam Asli |
| **Exp-C** | YOLOv8n Fine-Tuned | Dataset Low-Light + **Gamma Correction** ($\gamma=0.6$) | Citra Validasi Malam Asli |

> ⚠️ **Protokol Anti-Leakage:** Prapemrosesan CLAHE dan Gamma **HANYA** diterapkan pada split *Train*. Split *Validation* dan *Test* dibiarkan 100% murni tanpa augmentasi.

## 1. Verifikasi Akselerasi GPU & Instalasi Dependensi

In [ ]:
!nvidia-smi
!pip install -q ultralytics opencv-python-headless matplotlib seaborn pandas tqdm

## 2. Setup Dataset Low-Light v2 (Roboflow)
Unggah file dataset zip (`Low-light.v2i.yolov8.zip`) ke Google Drive atau langsung ke Google Colab, kemudian ekstrak di bawah ini.

In [ ]:
import os
import zipfile
from pathlib import Path

# Ekstraksi dataset jika masih dalam bentuk zip
zip_path = Path("Low-light.v2i.yolov8.zip")
dataset_dir = Path("dataset")

if zip_path.exists() and not dataset_dir.exists():
    print("Mengekstrak dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dataset_dir)
    print("Ekstraksi selesai!")
else:
    print("Direktori dataset sudah siap:", dataset_dir)

## 3. Implementasi Modul Prapemrosesan Adaptif (CLAHE & Gamma)
Membangun split training tereksitasi khusus untuk **Exp-B** (CLAHE) dan **Exp-C** (Gamma).

In [ ]:
import cv2
import numpy as np
import shutil
from tqdm import tqdm

def apply_clahe_lab(image, clip_limit=2.0, tile_grid=(8, 8)):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    enhanced_l = clahe.apply(l)
    merged = cv2.merge((enhanced_l, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

def apply_gamma(image, gamma=0.6):
    inv_gamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in range(256)]).astype("uint8")
    return cv2.LUT(image, table)

def build_variant(src_root, dst_root, method):
    if dst_root.exists():
        print(f"{dst_root} sudah ada. Lewati pembuatan.")
        return
    
    print(f"Membuat dataset varian {method.upper()}...")
    (dst_root / "train/images").mkdir(parents=True, exist_ok=True)
    (dst_root / "train/labels").mkdir(parents=True, exist_ok=True)
    
    # Transformasi HANYA gambar train
    train_imgs = list((src_root / "train/images").glob("*.jpg"))
    for p in tqdm(train_imgs, desc=f"Proses {method}"):
        img = cv2.imread(str(p))
        transformed = apply_clahe_lab(img) if method == "clahe" else apply_gamma(img)
        cv2.imwrite(str(dst_root / "train/images" / p.name), transformed)
        
    # Copy label train utuh
    for lbl in (src_root / "train/labels").glob("*.txt"):
        shutil.copy2(lbl, dst_root / "train/labels" / lbl.name)
        
    # Copy valid & test 100% ASLI (Anti-Leakage)
    for s in ["valid", "test"]:
        if (src_root / s).exists():
            shutil.copytree(src_root / s, dst_root / s, dirs_exist_ok=True)
            
    # Simpan data.yaml
    yaml_text = f"""path: {dst_root.resolve().as_posix()}
train: train/images
val: valid/images
test: test/images
nc: 2
names: ['Vehicle', 'person']
"""
    with open(dst_root / "data.yaml", "w") as f:
        f.write(yaml_text)
    print(f"Selesai membuat {dst_root}")

# Eksekusi pembuatan dataset
build_variant(Path("dataset"), Path("dataset_clahe"), method="clahe")
build_variant(Path("dataset"), Path("dataset_gamma"), method="gamma")

## 4. Pelatihan Model Eksperimen Terkontrol (50 Epochs, Seed=42)

In [ ]:
from ultralytics import YOLO

# Hyperparameter terstandarisasi untuk semua eksperimen
EPOCHS = 50
IMGSZ = 640
BATCH = 16
SEED = 42

# ==========================================
# Eksperimen A: Fine-Tuning Standar Low-Light
# ==========================================
print("\n🚀 Memulai Eksperimen A...")
model_a = YOLO("yolov8n.pt")
model_a.train(
    data="dataset/data.yaml",
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    seed=SEED,
    project="runs_comvis",
    name="Exp_A_Standard",
    save=True,
    device=0
)

# ==========================================
# Eksperimen B: Fine-Tuning + L-channel CLAHE
# ==========================================
print("\n🚀 Memulai Eksperimen B (CLAHE)...\n")
model_b = YOLO("yolov8n.pt")
model_b.train(
    data="dataset_clahe/data.yaml",
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    seed=SEED,
    project="runs_comvis",
    name="Exp_B_CLAHE",
    save=True,
    device=0
)

# ==========================================
# Eksperimen C: Fine-Tuning + Gamma Correction
# ==========================================
print("\n🚀 Memulai Eksperimen C (Gamma)...\n")
model_c = YOLO("yolov8n.pt")
model_c.train(
    data="dataset_gamma/data.yaml",
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    seed=SEED,
    project="runs_comvis",
    name="Exp_C_Gamma",
    save=True,
    device=0
)

## 5. Evaluasi & Komparasi Kinerja (Recall, Precision, mAP50)
Mengekstrak metrik evaluasi akhir dari ketiga eksperimen pada validation set.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Evaluasi pada dataset validasi murni
val_a = model_a.val(data="dataset/data.yaml", split="val")
val_b = model_b.val(data="dataset/data.yaml", split="val")
val_c = model_c.val(data="dataset/data.yaml", split="val")

# Rangkuman metrik
summary = pd.DataFrame({
    "Eksperimen": ["B0 (Zero-Shot Baseline)", "Exp-A (Standard Fine-Tuned)", "Exp-B (Proposed CLAHE)", "Exp-C (Gamma Correction)"],
    "Recall Person": [0.391, val_a.results_dict.get("metrics/recall(B)", 0.0), val_b.results_dict.get("metrics/recall(B)", 0.0), val_c.results_dict.get("metrics/recall(B)", 0.0)],
    "mAP@50": [0.380, val_a.results_dict.get("metrics/mAP50(B)", 0.0), val_b.results_dict.get("metrics/mAP50(B)", 0.0), val_c.results_dict.get("metrics/mAP50(B)", 0.0)]
})

print("\n=== TABEL PERBANDINGAN METRIK ===")
display(summary)

# Visualisasi Bar Chart Recall Pejalan Kaki
plt.figure(figsize=(10, 5))
plt.bar(summary["Eksperimen"], summary["Recall Person"], color=['gray', 'steelblue', 'green', 'orange'])
plt.title("Komparasi Recall Pejalan Kaki pada Kondisi Malam Hari", fontsize=13)
plt.ylabel("Recall Score (IoU=0.5)")
plt.ylim(0, 1.0)
for i, v in enumerate(summary["Recall Person"]):
    plt.text(i, v + 0.02, f"{v*100:.1f}%", ha='center', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig("komparasi_recall_pedestrian.png", dpi=300)
plt.show()